# Final models: training, TabPFN

Trains and saves TabPFN on the 60 summary features of B2 uncorrected. It is a comparison model in the focal-sampling comparison (`11a`, `12d`); the final classifier, HR B2 rotdaily, and the other models are trained in `10a_final_models_training`. TabPFN is in its own notebook because it needs the `tabpfn` environment.

Same windows and settings as the models in `10a_final_models_training` and the CV in `05c`: every labelled window the cross-validation used (the 2147 B2 windows in `indices_cv_fold_lookup.csv`), `TabPFNClassifier(random_state=42, balance_probabilities=True)` with default hyperparameters, no hold-out set. TabPFN's `.fit()` only stores the training rows as context, so "training" is fast. The model is saved to `models/final_tabpfn_b2_unc/`.

In [ ]:
import os
import time
import warnings
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier

In [ ]:
SEED = 42
N_WINDOWS = 2147  # labelled B2 windows in the fold lookup
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
fold_lookup = pd.read_csv(FOLD_LOOKUP_PATH)

TAB_DIR, TAB_NAME = "../models/final_tabpfn_b2_unc", "final_tabpfn_b2_unc"
ID_COLS = ["Ind_ID", "new_burst", "burst_id", "behaviour", "activity", "fold"]

# Same data preparation as 05c and the RF in 10a
features = pd.read_parquet(
    "../data/raw/features/Burst_2/annotated_features_burst_2_uncorrected.parquet"
)
m = fold_lookup.loc[fold_lookup["burst"] == "burst_2", ID_COLS]
df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(drop=True)
X_feat = df.drop(columns=ID_COLS)
assert X_feat.shape == (N_WINDOWS, 60), X_feat.shape

label_encoder = LabelEncoder()
y_feat = label_encoder.fit_transform(df["behaviour"])


def make_tabpfn():
    return TabPFNClassifier(random_state=SEED, balance_probabilities=True)

In [ ]:
# Check: this setup reproduces 05c's CV result for one fold
te = (df["fold"] == 0).to_numpy()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_pred = make_tabpfn().fit(X_feat[~te], y_feat[~te]).predict(X_feat[te])
ref = pd.read_csv(
    "../data/output/comparison_behaviour/behaviour_features_dl_tabpfn_metrics.csv"
)
ref = ref[
    (ref["burst"] == "Burst_2")
    & (ref["correction_type"] == "uncorrected")
    & (ref["model"] == "TabPFN")
    & (ref["fold"] == 0)
].iloc[0]
print(
    f"Fold 0 accuracy: {accuracy_score(y_feat[te], y_pred):.6f} (05c: {ref['accuracy']:.6f}); "
    f"f1_macro: {f1_score(y_feat[te], y_pred, average='macro', labels=np.arange(len(label_encoder.classes_)), zero_division=0):.6f} "
    f"(05c: {ref['f1_macro']:.6f})"
)

In [ ]:
tab = make_tabpfn()
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tab.fit(X_feat, y_feat)
tab_time = time.time() - t0

os.makedirs(TAB_DIR, exist_ok=True)
joblib.dump(tab, f"{TAB_DIR}/{TAB_NAME}_tabpfn_model.joblib")
joblib.dump(label_encoder, f"{TAB_DIR}/{TAB_NAME}_label_encoder.joblib")
pd.DataFrame(
    [
        {
            "model_name": TAB_NAME,
            "model_type": "TabPFN",
            "training_date": datetime.now().isoformat(),
            "data": "Burst_2 uncorrected",
            "n_windows": len(df),
            "n_features": X_feat.shape[1],
            "class_names": list(label_encoder.classes_),
            "balance_probabilities": True,
            "seed": SEED,
            "training_time_seconds": tab_time,
        }
    ]
).to_csv(f"{TAB_DIR}/{TAB_NAME}_training_metadata.csv", index=False)

# Reload check on the first 500 windows (TabPFN prediction is slower than RF's)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tab_reloaded = joblib.load(f"{TAB_DIR}/{TAB_NAME}_tabpfn_model.joblib")
    assert (tab_reloaded.predict(X_feat[:500]) == tab.predict(X_feat[:500])).all()
print(f"Saved TabPFN model to {TAB_DIR}; classes: {list(label_encoder.classes_)}")